# Semaine 2 — Recherche par le sens : embeddings + base vectorielle

**Objectif (énoncé de l'enseignant)** : découper les documents, calculer les embeddings, monter la base vectorielle, puis comparer recherche par mots-clés et recherche sémantique sur nos questions.

| Étape | Où |
|---|---|
| 1. Découper les PDF en passages (chunks) | `src/build_chunks.py` → `data/processed/chunks.jsonl` |
| 2. Calculer les embeddings + monter la base vectorielle | **ce notebook** |
| 3. Comparer mots-clés / sémantique sur les questions | `python src/compare_search.py` → `docs/evaluation/comparaison_recherche.md` |

Les embeddings sont calculés **localement** (modèle `sentence-transformers`), Groq n'est pas utilisé ici : il ne servira qu'à rédiger la réponse finale.

## 0. Préparation

Sur **Google Colab** : importez le dossier `COGNIVEX` (zip), puis décommentez les 2 lignes ci-dessous.
En local : `pip install -r requirements.txt` une seule fois, depuis la racine du projet.

In [2]:
import sys
from pathlib import Path

# On remonte jusqu'à trouver src/rag_config.py (marche depuis COGNIVEX/ ou COGNIVEX/notebooks/)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "rag_config.py").exists()), None)
if ROOT is None:
    # Recherche locale dans le dossier extrait si le CWD ne permet pas de remonter directement
    config_paths = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        try:
            config_paths.extend(base.rglob("src/rag_config.py"))
        except PermissionError:
            pass

    if config_paths:
        ROOT = config_paths[0].parent.parent

    if ROOT is None:
        raise FileNotFoundError(
            "Extrayez puis ouvrez le notebook depuis le dossier COGNIVEX "
            "(C:\\...\\COGNIVEX\\notebooks\\recherche.ipynb)."
        )
sys.path.insert(0, str(ROOT / "src"))
print("Racine du projet :", ROOT)

from rag_config import (COLLECTION_NAME, EMBEDDING_MODEL, TOP_K, VECTOR_DB_DIR,
                        charger_chunks, passage_prefix, query_prefix, texte_indexe)
print("Modèle d'embeddings :", EMBEDDING_MODEL)

Racine du projet : c:\Users\Hp\Desktop\COGNIVEX
Modèle d'embeddings : intfloat/multilingual-e5-small


## 1. Charger les chunks

`charger_chunks()` lit `data/processed/chunks.jsonl` (produit par `build_chunks.py`), retire le bruit d'impression du navigateur (dates, URL de pied de page) et donne un identifiant unique à chaque chunk.

In [3]:
from collections import Counter

chunks = charger_chunks()
print(f"{len(chunks)} chunks chargés\n")
for dci, n in sorted(Counter(c["dci"] for c in chunks).items()):
    print(f"{dci:40s} {n:5d} chunks")

2045 chunks chargés

AMOXICILLINE                               550 chunks
AMOXICILLINE + ACIDE CLAVULANIQUE          204 chunks
CÉFIXIME                                   403 chunks
CÉFUROXIME                                 199 chunks
CÉLÉCOXIB                                  145 chunks
NÉFOPAM                                     88 chunks
PARACÉTAMOL + CODÉINE                      216 chunks
TRAMADOL                                   240 chunks


Exemple de texte **réellement indexé** : on ajoute devant chaque chunk son contexte (DCI | produit | type de document | section), ce qui aide la recherche quand le passage seul ne nomme pas le médicament.

In [4]:
exemple = chunks[len(chunks) // 2]
print({k: v for k, v in exemple.items() if k != "text"})
print()
print(texte_indexe(exemple))

{'chunk_id': '01026-CÉFIXIME_À vérifier_74_2', 'dci': 'CÉFIXIME', 'produit': '', 'doc_type': 'Autre document PDF', 'fichier': 'data/corpus/céfixime/Article__CEFIXIME.pdf', 'page': 74, 'section': ''}

CÉFIXIME | Autre document PDF
made empiric choices guided by an antibiogram or not guided by an antibiogram. The question of whether an antibiogram is relevant and helpful to choosing an empiric antibiotic treatment regimen for an individual patient with cUTI is an open one. The modeling strategy used in determining the antibiogram susceptibility thresholds to guide empiric antibiotic choice in cUTI patients with sepsis could be tested in a real-world study to determine the value of such thresholds at improving clinical outcomes. Research on how to use rapid molecular testing to identify urinary organisms and susceptibility at the point of care is needed, as injudicious use of such tests may drive overtreatment and overly broad antibiotic use. Supplementary Data Supplementary materials are

## 2. Calculer les embeddings

Un **embedding** transforme un texte en une liste de nombres (ici 384) : deux textes de sens proche donnent des vecteurs proches. Le modèle `multilingual-e5-small` comprend le français et accepte 512 tokens (nos chunks de 120 mots passent sans être tronqués). Il demande le préfixe `passage: ` côté documents et `query: ` côté questions.

> La 1re exécution télécharge le modèle (~470 Mo, internet nécessaire).

In [5]:
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

modele = SentenceTransformer(EMBEDDING_MODEL)

Note: you may need to restart the kernel to use updated packages.


c:\Users\Hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 584.47it/s]


In [6]:
print("Modèle chargé avec succès !")
print(type(modele))

Modèle chargé avec succès !
<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [7]:
import time

textes = [passage_prefix() + texte_indexe(c) for c in chunks]

t0 = time.time()
# normalize_embeddings=True : vecteurs de norme 1 -> la distance cosinus est correcte
embeddings = modele.encode(textes, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(f"\n{embeddings.shape[0]} embeddings calculés en {time.time() - t0:.0f} s — dimension {embeddings.shape[1]}")

Batches: 100%|██████████| 32/32 [08:21<00:00, 15.67s/it]


2045 embeddings calculés en 502 s — dimension 384


In [8]:
import numpy as np

print("Norme du 1er vecteur :", round(float(np.linalg.norm(embeddings[0])), 4), "(1.0 = normalisé)")
print("8 premières valeurs  :", np.round(embeddings[0][:8], 3))

Norme du 1er vecteur : 1.0 (1.0 = normalisé)
8 premières valeurs  : [ 0.017 -0.039 -0.06  -0.081  0.071 -0.012  0.032  0.103]


## 3. Monter la base vectorielle (ChromaDB)

On stocke pour chaque chunk : son vecteur, son texte brut et ses métadonnées (DCI, produit, type de document, fichier, page, section) — de quoi **citer la source** (fichier + page) dans la réponse finale. La base est reconstruite à chaque exécution : pas de doublons.

In [9]:
import chromadb

VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass  # la collection n'existait pas encore

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine", "embedding_model": EMBEDDING_MODEL},
)

for i in range(0, len(chunks), 500):
    lot = chunks[i:i + 500]
    collection.add(
        ids=[c["chunk_id"] for c in lot],
        embeddings=embeddings[i:i + 500].tolist(),
        documents=[c["text"] for c in lot],
        metadatas=[{"dci": c["dci"], "produit": c["produit"], "doc_type": c["doc_type"],
                    "fichier": c["fichier"], "page": c["page"], "section": c["section"]} for c in lot],
    )

print(f"Base vectorielle prête : {VECTOR_DB_DIR.relative_to(ROOT)}  ({collection.count()} vecteurs)")

Base vectorielle prête : data\vectordb  (2045 vecteurs)


## 4. Test rapide : recherche par le sens

`chercher()` transforme la question en vecteur (préfixe `query: `) et retourne les chunks les plus proches. Les questions viennent de `docs/evaluation/questions_llm.md`.

In [10]:
def chercher(question, k=TOP_K):
    emb = modele.encode([query_prefix() + question], normalize_embeddings=True)
    res = collection.query(query_embeddings=emb.tolist(), n_results=k)
    for rang, (doc, meta, dist) in enumerate(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), 1):
        print(f"{rang}. similarité {1 - dist:.3f} | {meta['dci']} | {meta['produit'] or '-'} | "
              f"{meta['doc_type']} p.{meta['page']}")
        print("   ", " ".join(doc.split())[:200], "\n")


from compare_search import charger_questions

for q in charger_questions()[:3]:          # changez [:3] pour tester d'autres questions
    print("=" * 100)
    print(q["id"], "—", q["texte"], "\n")
    chercher(q["texte"], k=3)

Q1 — Quelle est la liste des formes pharmaceutiques et des dosages de l’association amoxicilline + acide clavulanique répertoriés dans la nomenclature pharmaceutique algérienne consultée ? 

1. similarité 0.898 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.18
    Patients insuffisants hépatiques L’association amoxicilline/acide clavulanique doit être utilisée avec prudence chez les patients insuffisants hépatiques et la fonction hépatique doit être surveillée  

2. similarité 0.897 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.2
    concernant l’utilisation appropriée des antibactériens. 4.2. Posologie et mode d'administration Posologie Les doses sont exprimées en quantité d’amoxicilline/acide clavulanique, sauf lorsqu’elles sont 

3. similarité 0.896 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.18
    à des hommes et des femmes sains, le sexe n’a pas d’incidence significa

## 5. Étape suivante : comparer avec la recherche par mots-clés

Depuis un terminal, à la racine du projet :

```
python src/compare_search.py
```

Le script pose **toutes les questions** de `docs/evaluation/questions_llm.md` aux deux méthodes (BM25 par mots-clés et recherche sémantique), sans réponse de référence, et écrit le rapport `docs/evaluation/comparaison_recherche.md` (extraits retrouvés, médicaments retrouvés, chunks communs aux deux méthodes).